# 10 · The Multi-Agent Mesh

*A supervisor that was never told what its workers can do.*

Notebook 08: MCP — an agent calls a tool in another process.
Notebook 09: A2A — an agent delegates to another agent.

This notebook puts both in one place. The **planner** is a supervisor with a
single tool catalogue of twelve entries. Six are MCP calls to tool servers.
Six are A2A delegations to specialist agents. They are bound to the model in
one `bind_tools()` call, and the model does not know which is which.

But the catalogue is not the interesting part. This is:

> **`services/planner/` contains zero hand-written knowledge of what any
> sub-agent can do.**

Every `delegate_to_<agent>` tool description is *fetched from that agent* at
startup and assembled from its Agent Card. Edit a card, restart that one
agent, and the supervisor's understanding of it changes — with no edit to the
supervisor. That is the thing a slide cannot show you, so we will do it live.

1. The catalogue: 12 tools, two protocols, one `bind_tools`.
2. Where each half's description comes from.
3. The synthesis, field by field.
4. Change a card → change the supervisor's mind.
5. What stays hardcoded, and why that is correct.
6. How the mesh stays coherent: shared state over MCP.
7. One delegation, end to end.
8. The trace tree that ties all of it together.

> **Prerequisite.** `docker compose up -d` in the repo root. This notebook is
> a *client and observer* of the running mesh — it does not reimplement it.

In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import json
import httpx

# The planner's config addresses its peers by Docker service name. We are
# outside that network, so point every URL at the host-mapped port BEFORE
# importing anything from services.planner (config reads os.environ at
# import time).
HOST_PORTS = {
    "FLIGHT_AGENT_A2A_URL":    8010,
    "HOTEL_AGENT_A2A_URL":     8011,
    "ITINERARY_AGENT_A2A_URL": 8012,
    "CRITIC_AGENT_A2A_URL":    8015,
    "TODO_AGENT_A2A_URL":      8016,
    "RESEARCH_AGENT_A2A_URL":  8018,
}
for var, port in HOST_PORTS.items():
    os.environ[var] = f"http://localhost:{port}/"

os.environ["MCP_TRIP_STATE_URL"] = "http://localhost:9015/mcp"
os.environ["MCP_PAYMENT_URL"]    = "http://localhost:9013/mcp"
os.environ["MCP_CALENDAR_URL"]   = "http://localhost:9014/mcp"

PHOENIX_URL = "http://localhost:6007"

def reachable(url: str) -> bool:
    try:
        httpx.get(url, timeout=3.0)
        return True
    except Exception:
        return False

checks = {
    "agents (6)":     all(reachable(f"http://localhost:{p}/health") for p in HOST_PORTS.values()),
    "mcp-trip-state": reachable("http://localhost:9015/health"),
    "phoenix":        reachable(PHOENIX_URL),
}
for label, ok in checks.items():
    print(f"  {'up  ' if ok else 'DOWN'}  {label}")

if not all(checks.values()):
    print("\n  Start the mesh:  docker compose up -d")

## Step 1 — One catalogue, two protocols

`services/planner/graph.py` builds its tool list in two lines:

```python
_a2a_tools = build_all_a2a_tools()      # 6 · discovered from Agent Cards
TOOLS = STATIC_TOOLS + _a2a_tools       # + 6 · hand-wrapped MCP calls
```

`build_all_a2a_tools()` does real network I/O at import time — it fetches six
Agent Cards before the planner can start. Running the next cell runs that
exact function, against the same live agents.

In [ ]:
from services.planner.tools import STATIC_TOOLS
from services.planner.dynamic_a2a_tools import build_all_a2a_tools

a2a_tools = build_all_a2a_tools()      # six live card fetches happen here
TOOLS = STATIC_TOOLS + a2a_tools

static_names = {t.name for t in STATIC_TOOLS}
print(f"The planner's catalogue · {len(TOOLS)} tools\n")
for t in TOOLS:
    origin = "mcp " if t.name in static_names else "a2a "
    print(f"  [{origin}] {t.name:28} {len(t.description):>5} chars of description")

Look at the description lengths. The MCP tools carry one or two lines —
a docstring somebody typed. The A2A tools carry over a thousand characters
each, and nobody typed any of it.

That asymmetry is the whole design. A deterministic tool needs a schema and
a sentence. An agent needs to be *explained* to whoever might delegate to it,
and the only party that can write that explanation honestly is the agent
itself.

## Step 2 — Where each half's description comes from

Side by side: a static tool's description is its Python docstring. A
delegate tool's description is assembled from a JSON document fetched over
HTTP a few seconds ago.

In [ ]:
by_name = {t.name for t in TOOLS}
static_sample = next(t for t in STATIC_TOOLS if t.name == "check_budget")
a2a_sample    = next(t for t in a2a_tools if t.name == "delegate_to_flight_agent")

print("── check_budget · hand-written in services/planner/tools.py ──")
print(static_sample.description)
print()
print("── delegate_to_flight_agent · first 5 lines, discovered ──")
for line in a2a_sample.description.splitlines()[:5]:
    print(line)
print(f"... ({len(a2a_sample.description)} chars total)")

## Step 3 — The synthesis, field by field

`_build_rich_description(card)` is twenty lines of string concatenation. It
takes the card's top-level `description`, then appends every skill's name,
description, and example briefs.

The result is the `description` field of a `StructuredTool` — which is
precisely the text Gemini reads when deciding whether to delegate. So the
sub-agent's own `examples` list ends up inside the supervisor's prompt.

In [ ]:
from services.planner.dynamic_a2a_tools import _fetch_agent_card, _build_rich_description

card = _fetch_agent_card(os.environ["FLIGHT_AGENT_A2A_URL"])
description = _build_rich_description(card)

print(description)

In [ ]:
# Prove it is derived, not written: every fragment below came off the card.
fragments = {
    "card.description":        card["description"][:60],
    "skill[0].name":           card["skills"][0]["name"],
    "skill[0].description":    card["skills"][0]["description"][:60],
    "skill[0].examples[0]":    card["skills"][0]["examples"][0],
    "skill[1].name":           card["skills"][1]["name"],
    "skill[1].examples[0]":    card["skills"][1]["examples"][0],
}
for label, frag in fragments.items():
    mark = "yes" if frag in description else "NO "
    print(f"  {mark}  {label:22} -> {frag[:52]}")

print(f"\nAll of it assembled by {_build_rich_description.__name__}(), "
      f"{len(description)} chars.")

## Step 4 — Change a card, change the supervisor's mind

The claim is that the supervisor's understanding of a worker is downstream of
that worker's card. The honest test is: change the card, restart that agent,
and watch the tool text change — without touching `services/planner/`.

We will not restart a container mid-notebook. Instead we do the same thing
deterministically: take the card we just fetched, edit a skill the way you
would edit `flight_agent/main.py`, and re-run the *real* synthesis function.
The output is what the planner would see after a restart.

In [ ]:
import copy

edited = copy.deepcopy(card)
edited["skills"][0]["description"] = (
    "Search-and-recommend mode. I return 12 flight options for ONE route + "
    "date, ranked, with a recommended pick and 2 backups. "
    "I now also flag flights with tight connections under 90 minutes."
)
edited["skills"][0]["examples"].append(
    "Search DEL to SIN on 2026-12-02 for 1 adult, avoid tight connections"
)

new_description = _build_rich_description(edited)

before = set(description.splitlines())
after  = set(new_description.splitlines())
print("lines the supervisor GAINED:\n")
for line in new_description.splitlines():
    if line and line not in before:
        print(f"  + {line}")

print(f"\n{len(description)} chars -> {len(new_description)} chars")
print("Files changed in services/planner/: none.")

That is capability discovery doing real work. The flight team can teach the
supervisor about tight connections by editing their own card — no
cross-team pull request, no supervisor redeploy beyond a restart.

The cost is that the supervisor's prompt is now partly written by six other
teams. Cards are prompt surface. A sloppy card is a prompt injection risk in
a trusted mesh, and a sloppy card in an *untrusted* mesh is worse.

## Step 5 — What stays hardcoded, and why that is correct

Discovery is not an argument for having no local knowledge. Three things
remain hand-written in the planner, and each is a genuine supervisor-level
concern rather than a fact about any single worker:

1. **The registry** — which agents exist at all.
2. **Orchestration order** — search before hold, check budget before commit.
   No single agent can know this; it is the shape of the whole workflow.
3. **The HITL gate** — the graph pauses before `capture_payment`. This is
   enforced by the *graph*, so no amount of model confidence can skip it.

In [ ]:
from services.planner.dynamic_a2a_tools import A2A_SUBAGENTS
import pathlib, re

print("1 · the registry · the only hand-coded fact about who exists\n")
for name, url in A2A_SUBAGENTS.items():
    print(f"    {name:16} {url}")

src = pathlib.Path("services/planner/graph.py").read_text()
print("\n3 · the HITL gate, straight out of graph.py\n")
for line in src.splitlines():
    stripped = line.strip()
    # the compile() argument itself, not the docstring diagram above it
    if stripped.startswith("interrupt_before="):
        print(f"    {stripped}")

prompt_src = pathlib.Path("services/planner/prompt.py").read_text()
order = [ln.strip() for ln in prompt_src.splitlines()
         if re.match(r"\s+\d\.\s+delegate_to_flight_agent", ln)]
print("\n2 · orchestration order · from the system prompt\n")
for ln in order[:3]:
    print(f"    {ln[:88]}")

## Step 6 — How a mesh stays coherent: shared state over MCP

Six agents in six processes. None of them shares memory with any other. So
how does the hotel agent know the flight already ate ₹1,05,000 of the budget?

It does not — and it must not have to. Budget lives in **`mcp-trip-state`**,
one MCP server holding the trip's state in Redis, keyed by `trip_id`. Any
participant reads and mutates the same record.

This is the part people skip when they build their first mesh, and it is why
their agents contradict each other. Delegation moves *work*; it does not move
*state*. State needs somewhere to live.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "trip_state": {"transport": "streamable_http", "url": os.environ["MCP_TRIP_STATE_URL"]}
})
ts_tools = {t.name: t for t in await client.get_tools()}
print("mcp-trip-state exposes:", ", ".join(sorted(ts_tools)))

def unwrap(raw):
    """MCP returns a list of TextContent blocks; each block's text is JSON."""
    if isinstance(raw, list) and raw and isinstance(raw[0], dict):
        return json.loads(raw[0]["text"])
    return raw if isinstance(raw, dict) else json.loads(raw)

TRIP = "nb10-demo"

state = unwrap(await ts_tools["set_budget"].ainvoke({"trip_id": TRIP, "limit_inr": 250_000}))
print(f"\n  set_budget      limit={state['limit_inr']:,}  remaining={state['remaining_inr']:,}")

chk = unwrap(await ts_tools["check_budget"].ainvoke(
    {"trip_id": TRIP, "proposed_amount_inr": 105_000, "category": "flight"}))
print(f"  check_budget    ok={chk['ok']}  proposed={chk['proposed']:,}")

state = unwrap(await ts_tools["commit_spend"].ainvoke(
    {"trip_id": TRIP, "amount_inr": 105_000, "category": "flight"}))
print(f"  commit_spend    spent={state['spent_inr']:,}  remaining={state['remaining_inr']:,}")

chk = unwrap(await ts_tools["check_budget"].ainvoke(
    {"trip_id": TRIP, "proposed_amount_inr": 160_000, "category": "hotel"}))
print(f"  check_budget    ok={chk['ok']}  reason={chk['reason']}")

The last line is the mesh being coherent. A *different* participant asked
about a *different* category and was refused — because the flight spend was
already committed to shared state.

Note what did **not** happen: the hotel agent was never told about the flight.
It asked the state server, which is the only component that needs to know.

## Step 7 — One delegation, end to end

Now the supervisor's actual move. `delegate_to_flight_agent` is a
`StructuredTool` whose coroutine calls `delegate_to_a2a_agent(url, brief)`.

From outside Docker the SDK path cannot resolve the card's advertised
hostname (notebook 09, Step 9), so we send the same A2A request the tool
would send, over the host port.

In [ ]:
import time

A2A_HEADERS = {"Content-Type": "application/json", "A2A-Version": "1.0"}

def a2a_send(base: str, brief: str) -> dict:
    envelope = {
        "jsonrpc": "2.0", "id": "nb-10", "method": "SendMessage",
        "params": {"message": {"messageId": "msg-nb-10", "role": "ROLE_USER",
                               "parts": [{"text": brief}]}},
    }
    r = httpx.post(base, headers=A2A_HEADERS, json=envelope,
                   timeout=httpx.Timeout(connect=5.0, read=180.0, write=30.0, pool=10.0))
    r.raise_for_status()
    return r.json()["result"]["task"]

BRIEF = "Search BLR to NRT on 2026-10-15 for 2 adults, prefer non-stop, under 150000 INR per leg"

t0 = time.time()
task = a2a_send(os.environ["FLIGHT_AGENT_A2A_URL"], BRIEF)
print(f"state {task['status']['state']} in {time.time() - t0:.1f}s")

prose = "".join(p.get("text", "") for p in task["status"]["message"]["parts"])
print(f"\ntext      · {prose[:150]}...")
for a in task.get("artifacts", []):
    rows = a["parts"][0]["data"].get("result", [])
    print(f"artifacts · {a['name']} · {len(rows)} rows for the UI cards")

One tool call from the supervisor's point of view. Underneath: an LLM call
in the flight agent, an MCP round trip to `mcp-airline`, ranking, and a
written recommendation.

That hiding is the supervisor pattern. The planner's transcript stays
readable — *tool started, tool finished* — while the real work happens a
layer down. Which is also why you need the next step.

## Step 8 — The trace tree

Every service calls `setup_observability()`, which instruments LangChain,
FastAPI (inbound) and httpx (outbound). The httpx instrumentation injects a
W3C `traceparent` header on the way out; the FastAPI instrumentation extracts
it on the way in. Because both the a2a-sdk client and the MCP client use
httpx underneath, that one pairing stitches every hop into a single trace.

The call we just made is in Phoenix now. Let us pull it back out and draw it.

In [ ]:
proj = httpx.get(f"{PHOENIX_URL}/v1/projects", timeout=10.0).json()["data"]
pid = next(p["id"] for p in proj if p["name"] == "agentic-ai-multiagent-orchestration")

spans = httpx.get(f"{PHOENIX_URL}/v1/projects/{pid}/spans",
                  params={"limit": 400}, timeout=20.0).json()["data"]
print(f"{len(spans)} spans in project 'agentic-ai-multiagent-orchestration'")

# The newest trace is the delegation we just sent.
newest = max(spans, key=lambda s: s["start_time"])
trace_id = newest["context"]["trace_id"]
in_trace = [s for s in spans if s["context"]["trace_id"] == trace_id]
print(f"newest trace {trace_id[:16]}… · {len(in_trace)} spans\n")

from datetime import datetime

children = {}
for s in in_trace:
    children.setdefault(s.get("parent_id"), []).append(s)
for kids in children.values():
    kids.sort(key=lambda s: s["start_time"])

# A root is any span whose parent is absent, or whose parent did not come
# back in this page of results. Without the second half you can silently
# get an empty tree.
present = {s["context"]["span_id"] for s in in_trace}
roots = sorted(
    (s for s in in_trace if not s.get("parent_id") or s["parent_id"] not in present),
    key=lambda s: s["start_time"],
)

def ms(s):
    t0 = datetime.fromisoformat(s["start_time"])
    t1 = datetime.fromisoformat(s["end_time"])
    return (t1 - t0).total_seconds() * 1000

# A2A's event queue emits a lot of sub-millisecond bookkeeping spans.
# Hide them so the agent loop is legible; drop this set to see everything.
NOISE = {"enqueue_event", "dequeue_event", "task_done", "tap"}

def draw(span, depth=0):
    name = span["name"].split(".")[-1][:52]
    if name not in NOISE:
        print(f"  {ms(span):8.0f}ms  {'   ' * depth}{'└─ ' if depth else ''}{name}")
        depth += 1
    if depth < 8:
        for kid in children.get(span["context"]["span_id"], []):
            draw(kid, depth)

for r in roots:
    draw(r)

print(f"\n  ({sum(1 for s in in_trace if s['name'].split('.')[-1] in NOISE)} "
      f"event-queue spans hidden)")

Read that tree and the agent loop is right there: **model → tools → model**.
The first `ChatGoogleGenerativeAI` span is the flight agent deciding to search;
`search_flights` is the MCP call; the second, longer LLM span is it reading
twelve options and writing the recommendation.

The bare `POST` spans are the httpx instrumentation catching outbound calls —
the two long ones are the Gemini API, the millisecond ones are MCP round trips
to `mcp-airline`. Those are the exact hops that carry the `traceparent` header,
which is why everything landed in one trace instead of six.

This is one A2A call seen from the inside: the JSON-RPC dispatcher accepting
the request, the executor building the graph, the LangGraph run, the LLM
calls, the MCP round trip.

Our notebook is not instrumented, so the trace begins where the request
*landed*. Send the same brief through the planner at
[localhost:8100](http://localhost:8100) and the tree gains the layers above:
planner → `delegate_to_flight_agent` → this whole subtree.

Open Phoenix at **[localhost:6007](http://localhost:6007)** to browse it with
timings and token counts.

## Recap

1. **One catalogue, two protocols.** 6 MCP tools + 6 A2A delegations, one
   `bind_tools()`. The model cannot tell them apart, and does not need to.
2. **MCP gives an agent capabilities. A2A lets agents delegate.** That is the
   one-line distinction to leave with.
3. **The supervisor holds no capability knowledge.** Every delegate tool's
   description is assembled from the sub-agent's own card at startup.
4. **Cards are prompt surface.** Edit a card, restart that agent, and the
   supervisor's prompt changes. Powerful, and a real trust boundary.
5. **Some things should stay hardcoded** — the registry, the orchestration
   order, the HITL gate. Those are supervisor concerns, not worker facts.
6. **Delegation moves work, not state.** Shared state lives in an MCP server
   so six stateless agents can stay consistent.
7. **Cross-service tracing is not optional here.** The supervisor pattern
   hides detail by design; `traceparent` propagation is how you get it back.

---

### Exercises

1. Do Step 4 for real: edit a skill description in
   `services/flight_agent/main.py`, `docker compose restart flight-agent
   planner`, and re-run Step 3. Confirm nothing in `services/planner/` changed.
2. Add a seventh agent. How many lines in the planner does it cost you?
   (`A2A_SUBAGENTS` plus its URL in `config.py` — and nothing else.)
3. Break a card on purpose: remove the `examples` from `search_flights`,
   restart, and ask the planner for flights. Does the delegation still route
   correctly? Examples earn their place or they do not.
4. Set a ₹50,000 budget, then ask for a ₹1L flight. Where does the refusal
   come from — the LLM's judgement, or `check_budget`? Which would you rather
   it was?
5. Watch the HITL gate: drive a booking to `capture_payment` in the UI and
   find the pause in `graph.py`. Try to get the model to skip it. It cannot;
   the interrupt is in the graph, not the prompt.